# SB1 worked example: one house
## REE 4301 / IE 5300 - Energy Systems Modeling

A small, complete version of what SB1 asks for, so you can see the shape before you build your own. It is deliberately short - your system will have different flows, but the same five moves.

The diagram at the end is the same kind of picture as the [LLNL energy flow charts](https://flowcharts.llnl.gov): sources on the left, what they were used for in the middle, and useful against rejected energy on the right.


In [ ]:
!pip install -q plotly


In [ ]:
import pandas as pd
import plotly.graph_objects as go


---
## 1. The boundary, in one sentence

*Everything that crosses the property line of one house over one year: electricity through the meter, gas through the pipe, and gasoline bought for the car parked there.*

Write yours the same way, and be as specific. Most of the trouble in this assignment comes from a boundary that was never actually stated.


## 2. What came in, in the units it was billed in

Bills do not arrive in one unit. Electricity is kWh, gas is therms or ccf, gasoline is gallons - so the conversion factors go in the table, where they can be checked.


In [ ]:
KWH_PER_THERM = 29.3      # 1 therm = 100,000 Btu
KWH_PER_GALLON = 33.7     # gasoline, lower heating value

inputs = pd.DataFrame([
    ['grid electricity', 10_000, 'kWh',    1.0,            'utility bill'],
    ['natural gas',         600, 'therms', KWH_PER_THERM,  'gas bill'],
    ['gasoline',            500, 'gallons', KWH_PER_GALLON, 'fuel receipts'],
], columns=['source', 'as billed', 'unit', 'kWh per unit', 'where from'])

inputs['kWh'] = inputs['as billed'] * inputs['kWh per unit']
print(inputs.to_string(index=False))
print(f"\ntotal energy in: {inputs['kWh'].sum():,.0f} kWh")


## 3. What it was used for, and how much of it did the job

Every conversion loses something. The efficiencies below are rough figures for a furnace, a car and a mixed electrical load - find better ones for your own system and say where you got them.


In [ ]:
efficiency = {'grid electricity': 0.90,   # lights, appliances, some heat
              'natural gas': 0.85,        # a decent furnace
              'gasoline': 0.25}           # tank to wheels

flows = inputs[['source', 'kWh']].copy()
flows['efficiency'] = flows['source'].map(efficiency)
flows['useful'] = flows['kWh'] * flows['efficiency']
flows['rejected'] = flows['kWh'] - flows['useful']

cols = {'kWh': 0, 'efficiency': 2, 'useful': 0, 'rejected': 0}
print(flows.round(cols).to_string(index=False))


## 4. Does it close?

In minus out. Here it closes exactly, because the efficiencies were *assumed* rather than measured - so the rejected column was calculated as the leftover.

Your balance will not close exactly, and that is the interesting part. Report the residual and say what it is.


In [ ]:
total_in = flows['kWh'].sum()
useful = flows['useful'].sum()
rejected = flows['rejected'].sum()

print(f'in        {total_in:>9,.0f} kWh')
print(f'useful    {useful:>9,.0f} kWh   {useful / total_in:.0%}')
print(f'rejected  {rejected:>9,.0f} kWh   {rejected / total_in:.0%}')
print(f'residual  {total_in - useful - rejected:>9,.0f} kWh')


---
## 5. Draw it

Widths come from the numbers above. Nothing is drawn by hand.


In [ ]:
labels = list(flows['source']) + ['useful energy', 'rejected energy']
i_useful, i_rejected = len(flows), len(flows) + 1

source_idx, target_idx, value = [], [], []
for i, row in flows.iterrows():
    source_idx += [i, i]
    target_idx += [i_useful, i_rejected]
    value += [row['useful'], row['rejected']]

fig = go.Figure(go.Sankey(
    node=dict(label=labels, pad=20, thickness=20,
              color=['#4C72B0', '#DD8452', '#937860', '#55A868', '#C44E52']),
    link=dict(source=source_idx, target=target_idx, value=value),
))
fig.update_layout(title_text='One house, one year (kWh)',
                  font_size=12, height=420)
fig.show()


### Read your own diagram

Look at the rejected block and work out which source is feeding most of it. Then check:


In [ ]:
share = (flows.set_index('source')['rejected'] / rejected).sort_values(
    ascending=False)
print('share of all rejected energy\n')
print(share.map('{:.0%}'.format).to_string())


> One source produces most of the waste in this house, and it is not the one with the largest bill. Why?

> This house comes out around 63% efficient. The LLNL chart puts the whole United States at roughly a third. Same physics, very different number - what is inside their boundary that is outside yours?


---
## What you do for your own system

Same five moves, your own numbers:

1. State the boundary in one sentence.
2. Table every flow in the unit it was measured in, with the conversion factor and the source alongside.
3. Split each flow into what did the job and what did not.
4. Report the residual, and say what it is rather than adjusting it away.
5. Generate the diagram from the table.

A car, a single power plant, a building and a small factory all work. Pick something you can find real numbers for.


*Before class: this balance was drawn around one house. If you drew it around the power station instead, which flows would move from outside the boundary to inside it?*
